# ***Project 1. Resume Analyzer***

In [ ]:
pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 9.6 MB/s eta 0:00:00


In [ ]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [ ]:
# Import required libraries
from google.colab import userdata
from openai import OpenAI

# Step 1: Fetch API key
api_key = userdata.get('hk')

# Step 2: Initialize Azure OpenAI client
client = OpenAI(
    api_key=api_key,
    base_url="https://kirtiaif.openai.azure.com/openai/v1"
)

In [ ]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_text}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [ ]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [ ]:
# Import required library
from azure.storage.blob import BlobServiceClient

# Step 1: Add your Azure Storage connection string
connection_string = "DefaultEndpointsProtocol=https;AccountName=kirstorage;AccountKey=fBjq3nosQX746wn+v78uHLLULJ090pwm8emAMsX0BfsSrL7C7STyY89xY89yr/60yXMN7PMpwelA+AStI/053w==;EndpointSuffix=core.windows.net"

try:
    # Step 2: Create Blob Service Client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)

    # Step 3: Define container name
    container_name = "resume-output"

    # Step 4: Get container client
    container_client = blob_service_client.get_container_client(container_name)

    # Step 5: Create container if it does not exist
    if not container_client.exists():
        print("Creating new container...")
        container_client.create_container()
    else:
        print("Container already exists")

    # Step 6: Define file name (blob name)
    blob_name = "result.txt"

    # Step 7: Get blob client
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    # Step 8: Data to upload
    data = "Test upload from Python (Colab)"

    # Step 9: Upload file to Azure Blob Storage
    blob_client.upload_blob(data, overwrite=True)

    print("File uploaded successfully!")

except Exception as e:
    # Step 10: Handle errors
    print("Upload failed!")
    print("Error details:", e)

Container already exists
File uploaded successfully!


In [ ]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [ ]:
blob_client = blob_service_client.get_blob_client(
    container="resume-output",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F6BED8BD6A2"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 5, 2, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': '0948de20-3d48-11f1-bc2a-0242ac1c000c',
 'request_id': '8e3ae192-a01e-0026-0f54-d1c37f000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 5, 2, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [ ]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_060449.txt


## ***Project 2. AI Research Paper Summarizer***

In [ ]:
# Import required libraries
from google.colab import userdata
from openai import OpenAI

# Step 1: Fetch API key
api_key = userdata.get('hk')

# Step 2: Initialize Azure OpenAI client
client = OpenAI(
    api_key=api_key,
    base_url="https://kirtiaif.openai.azure.com/openai/v1"
)

In [ ]:
# Step 1: Define input data (Research Paper)
paper_text = """
Artificial Intelligence (AI) is transforming industries by enabling machines to learn from data.
Deep learning, a subset of AI, uses neural networks with multiple layers.
This paper explores applications of AI in healthcare, finance, and education.
"""

# Step 2: Use Responses API for summarization
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Summarize the following research paper into 5-6 simple bullet points."
                    },
                    {
                        "type": "input_text",
                        "text": f"RESEARCH PAPER:\n{paper_text}"
                    }
                ]
            }
        ]
    )

    #  Step 3: Print summarized output
    print("📄 RESEARCH PAPER SUMMARY:\n")
    print(response.output_text)

except Exception as e:
    print(" Error:", e)

📄 RESEARCH PAPER SUMMARY:

- Artificial Intelligence (AI) helps machines learn from data and improve their performance.
- Deep learning is a type of AI that uses complex neural networks with many layers.
- The research paper discusses how AI is used in healthcare, finance, and education.
- AI can assist in medical diagnosis, automate financial processes, and personalize learning experiences.
- The paper highlights the growing impact of AI across different industries.


In [ ]:
try:
    # Step 1: Get response text safely
    output_item = response.output[0].content[0]

    # Handle both dict and object formats
    if isinstance(output_item, dict):
        summary_text = output_item.get("text", "")
    else:
        summary_text = output_item.text

    # Step 2: Print clean summary
    print(" SUCCESS: SUMMARY GENERATED \n")
    print(summary_text)

except Exception as e:
    print(" Parsing Error:", e)
    print("Actual content type:", type(response.output[0].content[0]))

 SUCCESS: SUMMARY GENERATED 

- Artificial Intelligence (AI) helps machines learn from data and improve their performance.
- Deep learning is a type of AI that uses complex neural networks with many layers.
- The research paper discusses how AI is used in healthcare, finance, and education.
- AI can assist in medical diagnosis, automate financial processes, and personalize learning experiences.
- The paper highlights the growing impact of AI across different industries.


In [ ]:
# Import required library
from azure.storage.blob import BlobServiceClient

# Step 1: Use NEW secure connection string
connection_string = "DefaultEndpointsProtocol=https;AccountName=kirstorage;AccountKey=fBjq3nosQX746wn+v78uHLLULJ090pwm8emAMsX0BfsSrL7C7STyY89xY89yr/60yXMN7PMpwelA+AStI/053w==;EndpointSuffix=core.windows.net"

try:
    # Assume: summary already generated from AI
    summary_text = response.output_text   # AI summarizer ka output

    # Step 2: Create Blob Service Client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)

    # Step 3: Define container (project specific)
    container_name = "research-summary"

    #  Step 4: Get container client
    container_client = blob_service_client.get_container_client(container_name)

    #  Step 5: Create container if not exists
    if not container_client.exists():
        print("Creating new container...")
        container_client.create_container()
    else:
        print(" Container already exists")

    # Step 6: Define file name
    blob_name = "paper_summary.txt"

    # Step 7: Create blob client
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    #  Step 8: Upload AI-generated summary
    blob_client.upload_blob(summary_text, overwrite=True)

    print(" Summary uploaded successfully to Azure Blob Storage!")

except Exception as e:
    # Step 9: Error handling
    print(" Upload failed!")
    print("Error details:", e)

Creating new container...
 Summary uploaded successfully to Azure Blob Storage!


In [ ]:
import datetime

# Step: Create a unique file name using current date & time
file_name = f"paper_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

print("Generated file name:", file_name)

Generated file name: paper_summary_20260421_061851.txt


In [ ]:
# Step: Get blob client using updated container and dynamic file name
blob_client = blob_service_client.get_blob_client(
    container="research-summary",   # Project-specific container
    blob=file_name                 # Dynamic file name (timestamp based)
)

#  Step: Upload AI-generated summary to Azure Blob Storage
blob_client.upload_blob(response.output_text, overwrite=True)

print(" Summary file uploaded successfully!")

 Summary file uploaded successfully!


In [ ]:
# Step: Confirm file storage with clear message
print(f" Summary successfully stored as: {file_name}")

 Summary successfully stored as: paper_summary_20260421_061851.txt


# ***Project 3. AI Medical Report Interpreter***

In [ ]:
# Import required libraries
from google.colab import userdata
from openai import OpenAI

# Step 1: Fetch API key
api_key = userdata.get('hk')

# Step 2: Initialize Azure OpenAI client
client = OpenAI(
    api_key=api_key,
    base_url="https://kirtiaif.openai.azure.com/openai/v1"
)

In [ ]:
# Step 1: Define input data (Medical Report)
medical_report = """
Patient Name: John Doe
Age: 45
Blood Pressure: 150/95 mmHg (High)
Cholesterol: 240 mg/dL (High)
Blood Sugar: 180 mg/dL (High)
Diagnosis: Risk of hypertension and diabetes.
Doctor Recommendation: Lifestyle changes, regular exercise, low-sugar diet, and medication.
"""

# Step 2: Use Responses API for interpretation
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Analyze the following medical report and explain it in simple terms with key points and health advice."
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_report}"
                    }
                ]
            }
        ]
    )

    # Step 3: Print interpreted output
    print("🩺 MEDICAL REPORT INTERPRETATION:\n")
    print(response.output_text)

except Exception as e:
    print("Error:", e)

🩺 MEDICAL REPORT INTERPRETATION:

Here’s a simple summary and explanation of the medical report:

**Key Points:**

1. **Blood Pressure:** 150/95 mmHg  
   - *What it means:* This is higher than normal (normal is around 120/80 mmHg). This is called high blood pressure or hypertension.

2. **Cholesterol:** 240 mg/dL  
   - *What it means:* This is higher than the recommended level (ideal is below 200 mg/dL). High cholesterol increases the risk of heart disease.

3. **Blood Sugar:** 180 mg/dL  
   - *What it means:* This is higher than normal. Normal fasting blood sugar is usually below 100 mg/dL. This suggests risk for diabetes.

4. **Diagnosis:**  
   - *Risk of hypertension and diabetes:* Based on these results, John is at increased risk for high blood pressure and diabetes.

5. **Doctor’s Recommendations:**  
   - Make healthy lifestyle changes  
   - Exercise regularly  
   - Eat a low-sugar diet  
   - Take prescribed medication

---

**Health Advice:**

- **Eat Healthily:** Choose 

In [ ]:
try:
    # Step 1: Get response text safely
    output_item = response.output[0].content[0]

    # Handle both dict and object formats
    if isinstance(output_item, dict):
        interpretation_text = output_item.get("text", "")
    else:
        interpretation_text = output_item.text

    # Step 2: Print clean interpretation
    print("🩺 SUCCESS: MEDICAL REPORT INTERPRETED\n")
    print(interpretation_text)

except Exception as e:
    print("Parsing Error:", e)
    print("Actual content type:", type(response.output[0].content[0]))

🩺 SUCCESS: MEDICAL REPORT INTERPRETED

Here’s a simple summary and explanation of the medical report:

**Key Points:**

1. **Blood Pressure:** 150/95 mmHg  
   - *What it means:* This is higher than normal (normal is around 120/80 mmHg). This is called high blood pressure or hypertension.

2. **Cholesterol:** 240 mg/dL  
   - *What it means:* This is higher than the recommended level (ideal is below 200 mg/dL). High cholesterol increases the risk of heart disease.

3. **Blood Sugar:** 180 mg/dL  
   - *What it means:* This is higher than normal. Normal fasting blood sugar is usually below 100 mg/dL. This suggests risk for diabetes.

4. **Diagnosis:**  
   - *Risk of hypertension and diabetes:* Based on these results, John is at increased risk for high blood pressure and diabetes.

5. **Doctor’s Recommendations:**  
   - Make healthy lifestyle changes  
   - Exercise regularly  
   - Eat a low-sugar diet  
   - Take prescribed medication

---

**Health Advice:**

- **Eat Healthily:** Ch

In [ ]:
# Import required library
from azure.storage.blob import BlobServiceClient

#
connection_string = "DefaultEndpointsProtocol=https;AccountName=kirstorage;AccountKey=fBjq3nosQX746wn+v78uHLLULJ090pwm8emAMsX0BfsSrL7C7STyY89xY89yr/60yXMN7PMpwelA+AStI/053w==;EndpointSuffix=core.windows.net"

try:
    # Step 2: Get AI interpretation output
    interpretation_text = response.output_text   # AI medical interpreter output

    # Step 3: Create Blob Service Client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)

    # Step 4: Define container (project specific)
    container_name = "medical-report-output"

    # Step 5: Get container client
    container_client = blob_service_client.get_container_client(container_name)

    # Step 6: Create container if not exists
    if not container_client.exists():
        print("Creating new container...")
        container_client.create_container()
    else:
        print("Container already exists")

    # Step 7: Define file name (dynamic recommended)
    import datetime
    file_name = f"medical_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

    # Step 8: Create blob client
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=file_name
    )

    # Step 9: Upload AI-generated interpretation
    blob_client.upload_blob(interpretation_text, overwrite=True)

    print(f"🩺 Medical report uploaded successfully as: {file_name}")

except Exception as e:
    # Step 10: Error handling
    print("Upload failed!")
    print("Error details:", e)

Container already exists
🩺 Medical report uploaded successfully as: medical_report_20260421_075605.txt


In [ ]:
import datetime

# Step: Create a unique file name using current date & time
file_name = f"medical_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

print("Generated file name:", file_name)

Generated file name: medical_report_20260421_075607.txt


In [ ]:
# Step: Get blob client using updated container and dynamic file name
blob_client = blob_service_client.get_blob_client(
    container="medical-report-output",   # Project-specific container
    blob=file_name                      # Dynamic file name (timestamp based)
)

# Step: Upload AI-generated medical interpretation
blob_client.upload_blob(response.output_text, overwrite=True)

print(" Medical report uploaded successfully!")

 Medical report uploaded successfully!


In [ ]:
# Step: Confirm file storage with clear message
print(f" Medical report successfully stored as: {file_name}")

 Medical report successfully stored as: medical_report_20260421_075607.txt
